# Stage 3: NLOS Ranging Error Prediction (BERT)
## Multi-Model Pipeline — Distance Correction via Single-Bounce Exploitation

**Purpose**: Given an NLOS signal where the single-bounce path is identifiable (Stage 2 = Correctable),
predict the **ranging error** in meters — how much the hardware's reported distance overshoots the true TX→RX distance.

**Target**: `ranging_error = Distance_hardware - d_direct` (per-sample, varies due to channel conditions and noise)

**Correction**: `d_corrected = d_hardware - predicted_error`

**Architecture**: Frozen BERT_Classifier encoder → 64-dim Transformer embeddings → Random Forest Regressor

**Data filtering**: Only correctable samples (bounce dominance ≥ threshold AND ≤ 2 peaks) — same mixture filter as Stage 2 labels.

**Pipeline**: Stage 1 (BERT → LOS/NLOS) → Stage 2 (BERT embeddings → RF → single-bounce identification) → Stage 3 (BERT embeddings → RF → ranging error)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    # ROI alignment (same as Stage 1/2)
    "search_start": 740,
    "search_end": 810,
    # Peak detection — morphological CIR quality (same as Stage 2)
    "peak_prominence": 0.20,
    "peak_min_distance": 5,
    "dominant_path_max_peaks": 2,
    # Geometric bounce dominance (same as Stage 2)
    "bounce_search_window": 3,
    "dominance_threshold": 0.50,
    # Random Forest Regressor
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    # Split
    "test_ratio": 0.30,
    "seed": 42,
}

np.random.seed(CONFIG["seed"])
print(f"Config: {CONFIG}")
print(f"\nTarget: ranging_error = Distance_hardware - d_direct (meters)")
print(f"Filter: correctable only (bounce dominance >= {CONFIG['dominance_threshold']:.0%} AND num_peaks <= {CONFIG['dominant_path_max_peaks']})")
print(f"Model: 64-dim BERT embeddings -> RF Regressor ({CONFIG['n_estimators']} trees)")

---
## Section 2: Data Loading & Correctable Filtering

1. Load NLOS samples from the combined dataset
2. Compute bounce dominance + peak count (same mixture criteria as Stage 2)
3. **Filter**: keep only correctable samples (both conditions met)
4. Compute per-sample ranging error target: `Distance - d_direct`

Stage 3 only trains on correctable samples because these are the signals where the single-bounce is identifiable —
Stage 3 would give unreliable predictions on challenging signals it was never trained on. Stage 2 gates inference to prevent this.

In [ ]:
# ── Ground-truth direct distances (TX→RX, geometric) ──
D_DIRECT_MAP = {
    "9.54m_nlos":  7.668,
    "12.79m_nlos": 8.81887,
    "8.91m_nlos":  6.1,
    "16.09m_nlos": 7.20139,
    "16.80m_nlos": 13.05986,
}

CIR_COLS = [f"CIR_{i}" for i in range(1016)]
RXPACC_COL = "RXPACC"
DIST_COL   = "Distance"
LABEL_COL  = "Label"
SRC_COL    = "Source_File"

def scenario_key(src):
    """Extract scenario prefix like '9.54m_nlos' from Source_File."""
    m = re.match(r"([\d.]+m_(?:nlos|los))", src)
    return m.group(1) if m else src

# ── ROI alignment ──
def get_roi_alignment(sig, search_start, search_end):
    """Find leading edge by back-tracking from peak in [search_start, search_end]."""
    roi = sig[search_start:search_end]
    peak_idx = search_start + np.argmax(roi)
    threshold = sig[peak_idx] * 0.05
    le = peak_idx
    while le > 0 and sig[le] > threshold:
        le -= 1
    return le + 1

# ── Peak count ──
def count_peaks(sig, leading_edge, config):
    """Count prominent peaks in CIR ROI (120 samples from leading edge)."""
    end = min(leading_edge + 120, len(sig))
    segment = sig[leading_edge:end]
    if len(segment) < 3:
        return 0
    peaks, _ = find_peaks(
        segment,
        prominence=config["peak_prominence"],
        distance=config["peak_min_distance"]
    )
    return len(peaks)

# ── Bounce dominance (amplitude ratio — matches Stage 2) ──
def compute_bounce_dominance(sig, leading_edge, bounce_path_idx, window):
    """Amplitude ratio: peak amplitude near bounce position / strongest peak in ROI."""
    roi_start = leading_edge
    roi_end = min(leading_edge + 120, len(sig))
    roi = sig[roi_start:roi_end]
    if len(roi) == 0:
        return 0.0

    max_overall = np.max(roi)
    if max_overall <= 0:
        return 0.0

    b_local = bounce_path_idx - roi_start
    bw_lo = max(0, b_local - window)
    bw_hi = min(len(roi), b_local + window + 1)
    bounce_window = roi[bw_lo:bw_hi]
    if len(bounce_window) == 0:
        return 0.0

    max_bounce = np.max(bounce_window)
    return float(max_bounce / max_overall)

# ── Load & filter ──
def load_correctable_nlos(filepath):
    """Load NLOS samples, apply mixture filter, keep only correctable, compute ranging error."""
    df = pd.read_csv(filepath)
    nlos_df = df[df[LABEL_COL] == 1].reset_index(drop=True)
    print(f"Total NLOS samples: {len(nlos_df)}")

    raw_sigs = nlos_df[CIR_COLS].values.astype(np.float64)
    rxpacc = nlos_df[RXPACC_COL].values.astype(np.float64)
    rxpacc[rxpacc == 0] = 1
    raw_sigs = raw_sigs / rxpacc[:, None]

    scenarios = nlos_df[SRC_COL].apply(scenario_key).values
    d_hardware = nlos_df[DIST_COL].values.astype(np.float64)

    # Map d_direct from ground-truth
    d_direct = np.array([D_DIRECT_MAP.get(s, np.nan) for s in scenarios])
    valid = ~np.isnan(d_direct)
    raw_sigs = raw_sigs[valid]
    scenarios = scenarios[valid]
    d_hardware = d_hardware[valid]
    d_direct = d_direct[valid]
    nlos_df = nlos_df.loc[nlos_df.index[valid]].reset_index(drop=True)

    # Bounce path index from d_bounce column (Source_File prefix)
    d_bounce = np.array([float(re.match(r"([\d.]+)m", s).group(1)) for s in scenarios])
    M_PER_IDX = 0.3003
    bounce_path_idx_arr = np.round(d_bounce / M_PER_IDX).astype(int)

    dom_thresh = CONFIG["dominance_threshold"]
    peak_thresh = CONFIG["dominant_path_max_peaks"]
    bw = CONFIG["bounce_search_window"]

    leading_edges = []
    bounce_dominance = []
    num_peaks = []
    for i in range(len(raw_sigs)):
        le = get_roi_alignment(raw_sigs[i], CONFIG["search_start"], CONFIG["search_end"])
        leading_edges.append(le)
        bd = compute_bounce_dominance(raw_sigs[i], le, bounce_path_idx_arr[i], bw)
        bounce_dominance.append(bd)
        np_ = count_peaks(raw_sigs[i], le, CONFIG)
        num_peaks.append(np_)

    leading_edges = np.array(leading_edges)
    bounce_dominance = np.array(bounce_dominance)
    num_peaks = np.array(num_peaks)

    is_correctable = (bounce_dominance >= dom_thresh) & (num_peaks <= peak_thresh)
    corr_idx = np.where(is_correctable)[0]
    print(f"Correctable: {len(corr_idx)} / {len(raw_sigs)} ({100*len(corr_idx)/len(raw_sigs):.1f}%)")

    raw_sigs_c = raw_sigs[corr_idx]
    scenarios_c = scenarios[corr_idx]
    d_hardware_c = d_hardware[corr_idx]
    d_direct_c = d_direct[corr_idx]
    leading_edges_c = leading_edges[corr_idx]
    ranging_error = d_hardware_c - d_direct_c

    print(f"Ranging error: mean={ranging_error.mean():.3f}m, std={ranging_error.std():.3f}m")
    for s in sorted(set(scenarios_c)):
        mask = scenarios_c == s
        print(f"  {s}: n={mask.sum()}, mean_error={ranging_error[mask].mean():.3f}m")

    return raw_sigs_c, scenarios_c, d_hardware_c, d_direct_c, leading_edges_c, ranging_error

# ── Run ──
DATA_PATH = "../dataset/channels/combined_uwb_dataset.csv"
raw_sigs, scenarios, d_hardware, d_direct, leading_edges, ranging_error = load_correctable_nlos(DATA_PATH)

---
## Section 3: Target Visualization

Visualize the ranging error distribution for correctable samples:
- Overall histogram
- Per-scenario breakdown
- d_hardware vs d_direct scatter

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Overall histogram
axes[0,0].hist(ranging_error, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0,0].axvline(ranging_error.mean(), color="red", ls="--", label=f"mean={ranging_error.mean():.3f}m")
axes[0,0].set_xlabel("Ranging Error (m)")
axes[0,0].set_ylabel("Count")
axes[0,0].set_title("Overall Ranging Error Distribution")
axes[0,0].legend()

# 2. Per-scenario box plot
df_viz = pd.DataFrame({"scenario": scenarios, "error": ranging_error})
df_viz.boxplot(column="error", by="scenario", ax=axes[0,1], rot=30)
axes[0,1].set_title("Per-Scenario Ranging Error")
axes[0,1].set_xlabel("")
axes[0,1].set_ylabel("Ranging Error (m)")
plt.sca(axes[0,1])
plt.title("Per-Scenario Ranging Error")

# 3. Scatter: d_hardware vs d_direct
for s in sorted(set(scenarios)):
    mask = scenarios == s
    axes[1,0].scatter(d_direct[mask], d_hardware[mask], alpha=0.3, s=10, label=s)
lims = [min(d_direct.min(), d_hardware.min())-0.5, max(d_direct.max(), d_hardware.max())+0.5]
axes[1,0].plot(lims, lims, "k--", alpha=0.5, label="ideal")
axes[1,0].set_xlabel("d_direct (m)")
axes[1,0].set_ylabel("d_hardware (m)")
axes[1,0].set_title("Hardware Distance vs True Distance")
axes[1,0].legend(fontsize=7)

# 4. Per-scenario summary
scens = sorted(set(scenarios))
counts = [np.sum(scenarios == s) for s in scens]
means  = [ranging_error[scenarios == s].mean() for s in scens]
x = np.arange(len(scens))
axes[1,1].bar(x, counts, alpha=0.6, label="count")
ax2 = axes[1,1].twinx()
ax2.plot(x, means, "ro-", label="mean error")
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(scens, rotation=30, fontsize=8)
axes[1,1].set_ylabel("Sample Count")
ax2.set_ylabel("Mean Ranging Error (m)")
axes[1,1].set_title("Per-Scenario Count & Mean Error")

plt.suptitle("Stage 3 — Correctable NLOS Ranging Error (BERT)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(f"\nTotal correctable samples: {len(ranging_error)}")

---
## Section 4: Frozen BERT Encoder — Embedding Extraction

Same frozen BERT_Classifier encoder as Stage 1/2. Extracts 64-dim time-averaged Transformer embeddings for correctable NLOS samples.

**Key**: No CLS token — uses average pooling across 60 CIR positions, then LayerNorm. No FP_AMPL conditioning.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

class BERT_Classifier(nn.Module):
    """
    Small Transformer Encoder for CIR classification — average pooling design.
    No CLS token — classifier sees time-averaged Transformer hidden states.
    No FP_AMPL conditioning — processes raw CIR only.
    """
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.2, max_seq_len=60):
        super().__init__()
        self.d_model = d_model

        # Input projection: scalar CIR value -> d_model
        self.input_proj = nn.Linear(input_size, d_model)

        # Learnable positional embeddings (60 CIR positions, no CLS)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Layer norm on pooled output
        self.norm = nn.LayerNorm(d_model)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid()
        )

    def _encode(self, x_seq):
        """Shared encoder: CIR -> transformer hidden states (B, 60, d_model)."""
        x = self.input_proj(x_seq)
        x = x + self.pos_embed[:, :x.size(1), :]
        h_all = self.transformer(x)
        return h_all

    def forward(self, x_seq, return_dynamics=False):
        h_all = self._encode(x_seq)
        h_avg = self.norm(h_all.mean(dim=1))
        pred = self.classifier(h_avg)
        if return_dynamics:
            return pred, h_all
        return pred

    def embed(self, x_seq):
        """Return d_model-dim embedding for Stage 2/3 compatibility."""
        h_all = self._encode(x_seq)
        return self.norm(h_all.mean(dim=1))

In [ ]:
# ── Load Stage 1 config ──
_saved = torch.load("stage1_bert_config.pt", map_location="cpu", weights_only=False)
STAGE1_CONFIG = _saved["config"]
EMBEDDING_DIM = STAGE1_CONFIG["d_model"]  # 64
print(f"Stage 1 config: d_model={STAGE1_CONFIG['d_model']}, nhead={STAGE1_CONFIG['nhead']}, "
      f"num_layers={STAGE1_CONFIG['num_layers']}, dim_ff={STAGE1_CONFIG['dim_feedforward']}")

# ── Instantiate & load frozen encoder ──
bert_encoder = BERT_Classifier(
    input_size=STAGE1_CONFIG["input_size"],
    d_model=STAGE1_CONFIG["d_model"],
    nhead=STAGE1_CONFIG["nhead"],
    num_layers=STAGE1_CONFIG["num_layers"],
    dim_feedforward=STAGE1_CONFIG["dim_feedforward"],
    dropout=STAGE1_CONFIG["dropout"],
    max_seq_len=STAGE1_CONFIG["total_len"],
).to(device)

checkpoint_path = "stage1_bert_best.pt"
bert_encoder.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
bert_encoder.eval()
for param in bert_encoder.parameters():
    param.requires_grad = False
print(f"Loaded frozen BERT encoder from {checkpoint_path}")
print(f"Embedding dim: {EMBEDDING_DIM}")

# ── Preprocess CIR for BERT (same as Stage 2) ──
def preprocess_cir_for_bert(sig, leading_edge):
    """Convert a single RXPACC-normalized CIR + leading edge to 60-sample window."""
    PRE = STAGE1_CONFIG["pre_crop"]
    TOTAL = STAGE1_CONFIG["total_len"]
    start = max(0, leading_edge - PRE)
    end = start + TOTAL
    if end > len(sig):
        end = len(sig)
        start = max(0, end - TOTAL)
    crop = sig[start:end]
    if len(crop) < TOTAL:
        crop = np.pad(crop, (0, TOTAL - len(crop)), mode="constant")
    local_min, local_max = np.min(crop), np.max(crop)
    rng = local_max - local_min
    crop = (crop - local_min) / rng if rng > 0 else np.zeros(TOTAL)
    return crop

# ── Extract 64-dim BERT embeddings ──
cir_sequences = []
for i in range(len(raw_sigs)):
    crop = preprocess_cir_for_bert(raw_sigs[i], leading_edges[i])
    cir_sequences.append(crop)

cir_tensor = torch.tensor(
    np.array(cir_sequences).reshape(-1, STAGE1_CONFIG["total_len"], 1),
    dtype=torch.float32
).to(device)

all_embeddings = []
with torch.no_grad():
    for i in range(0, len(cir_tensor), 256):
        batch_cir = cir_tensor[i:i+256]
        emb = bert_encoder.embed(batch_cir)  # (batch, 64)
        all_embeddings.append(emb.cpu().numpy())

embeddings = np.vstack(all_embeddings)
EMBEDDING_NAMES = [f"bert_emb_{i}" for i in range(EMBEDDING_DIM)]
FEATURE_DIM = EMBEDDING_DIM
print(f"Embeddings shape: {embeddings.shape}")
print(f"Feature dim: {FEATURE_DIM} (BERT Transformer embeddings only)")

---
## Section 5: Random Forest Regressor

**Architecture**: Frozen BERT_Classifier encoder → **64-dim Transformer embeddings** → Random Forest Regressor

- **Input**: 64-dim BERT embeddings (time-averaged self-attention representations, no FP conditioning)
- **Target**: Ranging error in meters (`Distance_hardware - d_direct`)
- **At inference**: No ground truth needed — the RF predicts error from embeddings alone, Stage 2 gates which samples reach here

In [ ]:
# ── 70/30 stratified split ──
X_train, X_test, y_train, y_test, scen_train, scen_test = train_test_split(
    embeddings, ranging_error, scenarios,
    test_size=CONFIG["test_ratio"],
    random_state=CONFIG["seed"],
    stratify=scenarios,
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train scenarios: {dict(zip(*np.unique(scen_train, return_counts=True)))}")
print(f"Test  scenarios: {dict(zip(*np.unique(scen_test, return_counts=True)))}")

# ── Train RF Regressor ──
rf_model = RandomForestRegressor(
    n_estimators=CONFIG["n_estimators"],
    max_depth=CONFIG["max_depth"],
    min_samples_split=CONFIG["min_samples_split"],
    min_samples_leaf=CONFIG["min_samples_leaf"],
    random_state=CONFIG["seed"],
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
print("RF Regressor trained.")

# ── Evaluate ──
y_pred_train = rf_model.predict(X_train)
y_pred_test  = rf_model.predict(X_test)

train_mae  = mean_absolute_error(y_train, y_pred_train)
test_mae   = mean_absolute_error(y_test, y_pred_test)
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2    = r2_score(y_test, y_pred_test)

print(f"\n{'='*50}")
print(f"  Train MAE:  {train_mae:.4f} m")
print(f"  Test  MAE:  {test_mae:.4f} m")
print(f"  Test  RMSE: {test_rmse:.4f} m")
print(f"  Test  R\u00b2:   {test_r2:.4f}")
print(f"{'='*50}")

# ── Per-scenario test MAE ──
print("\nPer-scenario test MAE:")
for s in sorted(set(scen_test)):
    mask = scen_test == s
    mae_s = mean_absolute_error(y_test[mask], y_pred_test[mask])
    print(f"  {s}: MAE={mae_s:.4f}m (n={mask.sum()})")

---
## Section 6: Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature importance (top 15)
importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[-15:]
axes[0,0].barh([EMBEDDING_NAMES[i] for i in top_idx], importances[top_idx], color="steelblue")
axes[0,0].set_xlabel("Importance")
axes[0,0].set_title("Top 15 BERT Embedding Dimensions")

# 2. Predicted vs Actual (by scenario)
for s in sorted(set(scen_test)):
    mask = scen_test == s
    axes[0,1].scatter(y_test[mask], y_pred_test[mask], alpha=0.4, s=15, label=s)
lims = [min(y_test.min(), y_pred_test.min())-0.2, max(y_test.max(), y_pred_test.max())+0.2]
axes[0,1].plot(lims, lims, "k--", alpha=0.5)
axes[0,1].set_xlabel("Actual Ranging Error (m)")
axes[0,1].set_ylabel("Predicted Ranging Error (m)")
axes[0,1].set_title("Predicted vs Actual (Test Set)")
axes[0,1].legend(fontsize=7)

# 3. Residual histogram
residuals = y_test - y_pred_test
axes[1,0].hist(residuals, bins=50, edgecolor="black", alpha=0.7, color="salmon")
axes[1,0].axvline(0, color="black", ls="--")
axes[1,0].set_xlabel("Residual (m)")
axes[1,0].set_ylabel("Count")
axes[1,0].set_title(f"Residuals (mean={residuals.mean():.4f}, std={residuals.std():.4f})")

# 4. Per-scenario MAE: before vs after correction
scens = sorted(set(scen_test))
x = np.arange(len(scens))
mae_before = []
mae_after  = []
for s in scens:
    mask = scen_test == s
    mae_before.append(np.abs(y_test[mask]).mean())  # |error| before correction
    mae_after.append(np.abs(y_test[mask] - y_pred_test[mask]).mean())  # |residual| after correction

w = 0.35
axes[1,1].bar(x - w/2, mae_before, w, label="Before correction", color="salmon", alpha=0.7)
axes[1,1].bar(x + w/2, mae_after, w, label="After correction", color="steelblue", alpha=0.7)
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(scens, rotation=30, fontsize=8)
axes[1,1].set_ylabel("MAE (m)")
axes[1,1].set_title("Per-Scenario: Before vs After Correction")
axes[1,1].legend()

plt.suptitle("Stage 3 Diagnostics — BERT RF Regressor", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 7: Unseen Dataset Evaluation

In [ ]:
UNSEEN_PATH = "../dataset/channels/unseen_dataset.csv"
if not os.path.exists(UNSEEN_PATH):
    print("Unseen dataset not found — skipping.")
else:
    unseen_raw_sigs, unseen_scenarios, unseen_d_hw, unseen_d_direct, unseen_le, unseen_re = \
        load_correctable_nlos(UNSEEN_PATH)

    if len(unseen_raw_sigs) == 0:
        print("No correctable NLOS in unseen dataset.")
    else:
        # Extract BERT embeddings
        unseen_cir_sequences = []
        for i in range(len(unseen_raw_sigs)):
            crop = preprocess_cir_for_bert(unseen_raw_sigs[i], unseen_le[i])
            unseen_cir_sequences.append(crop)

        unseen_cir_tensor = torch.tensor(
            np.array(unseen_cir_sequences).reshape(-1, STAGE1_CONFIG["total_len"], 1),
            dtype=torch.float32
        ).to(device)

        unseen_embeddings_list = []
        with torch.no_grad():
            for i in range(0, len(unseen_cir_tensor), 256):
                batch_cir = unseen_cir_tensor[i:i+256]
                emb = bert_encoder.embed(batch_cir)
                unseen_embeddings_list.append(emb.cpu().numpy())

        unseen_embeddings = np.vstack(unseen_embeddings_list)

        # Predict
        unseen_pred = rf_model.predict(unseen_embeddings)
        unseen_mae  = mean_absolute_error(unseen_re, unseen_pred)
        unseen_rmse = np.sqrt(mean_squared_error(unseen_re, unseen_pred))
        unseen_r2   = r2_score(unseen_re, unseen_pred)

        print(f"\n{'='*50}")
        print(f"  UNSEEN MAE:  {unseen_mae:.4f} m")
        print(f"  UNSEEN RMSE: {unseen_rmse:.4f} m")
        print(f"  UNSEEN R\u00b2:   {unseen_r2:.4f}")
        print(f"{'='*50}")

        print("\nPer-scenario unseen MAE:")
        for s in sorted(set(unseen_scenarios)):
            mask = unseen_scenarios == s
            mae_s = mean_absolute_error(unseen_re[mask], unseen_pred[mask])
            print(f"  {s}: MAE={mae_s:.4f}m (n={mask.sum()})")

In [ ]:
if os.path.exists(UNSEEN_PATH) and len(unseen_raw_sigs) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Predicted vs Actual (unseen)
    for s in sorted(set(unseen_scenarios)):
        mask = unseen_scenarios == s
        axes[0].scatter(unseen_re[mask], unseen_pred[mask], alpha=0.4, s=15, label=s)
    lims = [min(unseen_re.min(), unseen_pred.min())-0.2,
            max(unseen_re.max(), unseen_pred.max())+0.2]
    axes[0].plot(lims, lims, "k--", alpha=0.5)
    axes[0].set_xlabel("Actual Ranging Error (m)")
    axes[0].set_ylabel("Predicted Ranging Error (m)")
    axes[0].set_title("Predicted vs Actual (Unseen)")
    axes[0].legend(fontsize=7)

    # 2. Residual histogram (unseen)
    unseen_resid = unseen_re - unseen_pred
    axes[1].hist(unseen_resid, bins=40, edgecolor="black", alpha=0.7, color="salmon")
    axes[1].axvline(0, color="black", ls="--")
    axes[1].set_xlabel("Residual (m)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Unseen Residuals (mean={unseen_resid.mean():.4f})")

    # 3. Per-scenario before vs after (unseen)
    scens = sorted(set(unseen_scenarios))
    x = np.arange(len(scens))
    mb = [np.abs(unseen_re[unseen_scenarios == s]).mean() for s in scens]
    ma = [np.abs(unseen_resid[unseen_scenarios == s]).mean() for s in scens]
    w = 0.35
    axes[2].bar(x - w/2, mb, w, label="Before", color="salmon", alpha=0.7)
    axes[2].bar(x + w/2, ma, w, label="After", color="steelblue", alpha=0.7)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(scens, rotation=30, fontsize=8)
    axes[2].set_ylabel("MAE (m)")
    axes[2].set_title("Unseen: Before vs After Correction")
    axes[2].legend()

    plt.suptitle("Stage 3 Unseen Diagnostics — BERT", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Section 8: Save Artifacts

In [ ]:
# Save RF model
joblib.dump(rf_model, "stage3_nlos_bias_rf.joblib")
print("Saved: stage3_nlos_bias_rf.joblib")

# Save config metadata
joblib.dump({
    "config": CONFIG,
    "embedding_dim": EMBEDDING_DIM,
    "embedding_names": EMBEDDING_NAMES,
    "feature_dim": FEATURE_DIM,
    "stage1_config": STAGE1_CONFIG,
    "encoder_class": "BERT_Classifier",
    "fp_conditioning": False,
    "filter_strategy": "mixture_geometric_morphological",
    "dominance_threshold": CONFIG["dominance_threshold"],
    "bounce_search_window": CONFIG["bounce_search_window"],
    "dominant_path_max_peaks": CONFIG["dominant_path_max_peaks"],
    "target": "ranging_error = Distance_hardware - d_direct",
    "correction": "d_corrected = d_hardware - predicted_error",
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "note": "Stage 3 RF regressor on 64-dim BERT Transformer embeddings. "
           "Correctable NLOS only (bounce_dominance >= threshold AND num_peaks <= 2). "
           "Uses amplitude-ratio bounce dominance (matches Stage 2).",
}, "stage3_config.joblib")
print("Saved: stage3_config.joblib")

In [ ]:
print("=" * 60)
print("  STAGE 3 SUMMARY — BERT NLOS Ranging Error Prediction")
print("=" * 60)
print(f"  Encoder:         BERT_Classifier (frozen, no FP conditioning)")
print(f"  Embedding dim:   {EMBEDDING_DIM}")
print(f"  Model:           Random Forest Regressor ({CONFIG['n_estimators']} trees)")
print(f"  Target:          ranging_error = d_hardware - d_direct")
print(f"  Correction:      d_corrected = d_hardware - predicted_error")
print(f"  Train samples:   {len(X_train)}")
print(f"  Test samples:    {len(X_test)}")
print(f"  Test MAE:        {test_mae:.4f} m")
print(f"  Test RMSE:       {test_rmse:.4f} m")
print(f"  Test R\u00b2:         {test_r2:.4f}")
print(f"  Artifacts:       stage3_nlos_bias_rf.joblib, stage3_config.joblib")
print("=" * 60)